# GB grid analytics
This notebook reproduces the dashboard's descriptive analysis from local snapshots. Run `python scripts/analytics.py --refresh-db` from the project first. No notebook cell modifies the database. Historical error associations are not causal explanations.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'scripts'))
from analytics import patterns, price_analysis
from backtest import read_frame
report = json.loads((root / 'artifacts/analytics/latest.json').read_text())
intensity = read_frame(root / 'artifacts/analytics/intensity.csv')
print(report['source'], report['generated_at'])
report['patterns']['quality']

## Coverage and gaps
UTC half-hours define completeness. London local hours define usage patterns, including daylight-saving transitions. Missing periods are never filled with zero.

In [ ]:
analysis = patterns(intensity)
pd.DataFrame(analysis['gaps'])

In [ ]:
pd.DataFrame(analysis['monthly_hourly']).pivot(index='month', columns='hour', values='mean').round(1)

## Diagnose the 6–24 hour bias
Compare signed errors across horizons and weather groups. Check sample sizes before interpreting group differences. These groups are confounded by season and delivery hour; they do not identify why an error occurred.

In [ ]:
pd.DataFrame(report['errors']['horizons'])

In [ ]:
pd.DataFrame(report['errors']['weather']['wind'])

## Carbon versus price
The comparison uses four consecutive half-hours on complete UTC days only. It is a hindsight comparison using settled outcomes, not a forecast-based savings estimate.

In [ ]:
price_path = root / 'artifacts/analytics/matched-prices.csv'
price_results = price_analysis(read_frame(price_path)) if report['prices']['available'] and price_path.exists() else {'available': False}
{k:v for k,v in price_results.items() if k not in ('hourly', 'recent_windows')}